In [ ]:
import pandas as pd
import os
import sys

# 1. Define the file path
raw_path = '../data/raw/ethiopia_fi_unified_data.xlsx'

try:
    # 2. Load the Excel file specifically using your sheet names
    all_sheets = pd.read_excel(raw_path, sheet_name=None)
    
    # Assign data based on your specific Excel tab names
    df_main = all_sheets['ethiopia_fi_unified_data']
    df_links = all_sheets['Impact_sheet']
    
    print(f"✅ Successfully loaded 'ethiopia_fi_unified_data' ({len(df_main)} rows)")
    print(f"✅ Successfully loaded 'Impact_sheet' ({len(df_links)} rows)")

    # 3. Filter df_main to only include events
    # We use .copy() to avoid SettingWithCopy warnings later
    events = df_main[df_main['record_type'] == 'event'].copy()
    
    print(f"\nSummary:")
    print(f"- Events available for modeling: {len(events)}")
    print(f"- Impact Link relationships: {len(df_links)}")
    
    # 4. Show the columns in Impact_sheet so we can find the link ID
    print("\nColumns found in Impact_sheet:")
    print(df_links.columns.tolist())

except Exception as e:
    print(f"❌ Error: {e}")

# Preview the data
print("\n--- Impact Links Preview ---")
display(df_links.head(3))

In [ ]:
# --- Cell 2: Quantifying Qualitative Impacts ---

# 1. Define the numeric mapping for magnitudes
# These values represent estimated percentage point (pp) impacts
magnitude_map = {
    'high': 10.0,
    'medium': 5.0,
    'low': 2.0,
    'minimal': 0.5
}

def calculate_impact_score(row):
    """
    Combines direction and magnitude into a single numeric score.
    Example: 'increase' + 'high' -> +10.0
    """
    # Clean the input strings
    mag = str(row.get('impact_magnitude', 'low')).lower().strip()
    direction = str(row.get('impact_direction', 'increase')).lower().strip()
    
    # Get numeric magnitude
    val = magnitude_map.get(mag, 2.0) # Default to 2.0 if unknown
    
    # Determine sign based on direction
    # Handles both 'increase'/'decrease' and 'positive'/'negative'
    if direction in ['increase', 'positive']:
        return val
    elif direction in ['decrease', 'negative']:
        return -val
    else:
        return val

# 2. Apply the quantification function
df_links['impact_score'] = df_links.apply(calculate_impact_score, axis=1)

# 3. Handle Lags (Time it takes for an event to show impact)
# Fill missing lag months with a default of 6
df_links['lag_months'] = pd.to_numeric(df_links['lag_months'], errors='coerce').fillna(6).astype(int)

# 4. Clean IDs to ensure perfect matching in the next step
df_links['parent_id'] = df_links['parent_id'].astype(str).str.strip()
events['record_id'] = events['record_id'].astype(str).str.strip()

print("✅ Impacts quantified and IDs cleaned.")
display(df_links[['parent_id', 'impact_direction', 'impact_magnitude', 'impact_score', 'lag_months']].head())

In [ ]:
# --- Cell 3: Building the Association Matrix ---
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Merge Links with Event Names
# We connect df_links (parent_id) to events (record_id)
impact_matrix_df = df_links.merge(
    events[['record_id', 'indicator']], 
    left_on='parent_id', 
    right_on='record_id', 
    how='inner',
    suffixes=('_link', '_event')
)

# 2. Pivot the data into a Matrix
# Rows = Event Names (indicator_event)
# Columns = Indicators being affected (related_indicator)
association_matrix = impact_matrix_df.pivot_table(
    index='indicator_event',        
    columns='related_indicator',    
    values='impact_score',
    aggfunc='mean'                  
).fillna(0)

# 3. Plotting the Heatmap
plt.figure(figsize=(12, 8))
sns.set_theme(style="white")

sns.heatmap(
    association_matrix, 
    annot=True, 
    cmap="RdYlGn", 
    center=0, 
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={'label': 'Impact (pp)'}
)

plt.title("Event-Indicator Association Matrix: Ethiopia", fontsize=15, pad=20)
plt.ylabel("Catalyst Events")
plt.xlabel("Indicators Affected")
plt.xticks(rotation=45, ha='right')

# Save the figure
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/association_matrix.png', bbox_inches='tight', dpi=300)

plt.show()

print(f"✅ Association Matrix built with {association_matrix.shape[0]} events and {association_matrix.shape[1]} indicators.")

In [28]:
# --- Cell 4: Calibration and Final Weights Export ---

# 1. Define our validation target
# Based on Findex 2021-2024 data, actual growth was 3.0 percentage points
observed_growth = 3.0
target_indicator = 'ACC_OWNERSHIP'

print(f"--- Calibration Validation: {target_indicator} ---")

if target_indicator in association_matrix.columns:
    # We look at events that happened between 2021-2023
    historical_events = ['Telebirr Launch', 'Fayda Digital ID Program Rollout']
    
    # Calculate modeled lift for these events
    theoretical_lift = 0
    for event in historical_events:
        if event in association_matrix.index:
            theoretical_lift += association_matrix.loc[event, target_indicator]
    
    print(f"Theoretical Model Lift: {theoretical_lift}%")
    print(f"Actual Observed Growth: {observed_growth}%")
    
    # 2. Calculate the Calibration Factor (The Multi-homing Factor)
    # If theoretical is 15% and actual is 3%, our factor is 0.2
    calibration_factor = observed_growth / theoretical_lift if theoretical_lift > 0 else 1.0
    print(f"Calculated Calibration Factor: {calibration_factor:.2f}")
    
    # 3. Apply Calibration to the Matrix
    # We apply this specifically to the 'Access' indicator to keep forecasts grounded
    calibrated_weights = association_matrix.copy()
    calibrated_weights[target_indicator] = calibrated_weights[target_indicator] * calibration_factor
    
    # 4. Export the weights for Task 4 Forecasting
    os.makedirs('../data/processed', exist_ok=True)
    calibrated_weights.to_csv('../data/processed/calibrated_impact_weights.csv')
    
    print(f"\n✅ SUCCESS: Calibrated weights saved to 'data/processed/calibrated_impact_weights.csv'")
    print(f"💡 INSIGHT: A factor of {calibration_factor:.2f} means only about {calibration_factor*100:.0f}% "
          "of new mobile money registrations result in a 'New Account' in surveys. The rest is multi-homing.")

else:
    print(f"⚠️ Indicator '{target_indicator}' not found in matrix. Saving raw weights.")
    association_matrix.to_csv('../data/processed/impact_weights_raw.csv')

--- Calibration Validation: ACC_OWNERSHIP ---
Theoretical Model Lift: 15.0%
Actual Observed Growth: 3.0%
Calculated Calibration Factor: 0.20

✅ SUCCESS: Calibrated weights saved to 'data/processed/calibrated_impact_weights.csv'
💡 INSIGHT: A factor of 0.20 means only about 20% of new mobile money registrations result in a 'New Account' in surveys. The rest is multi-homing.
